In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import numpy as np
import cv2
import os


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [12]:
!nvidia-smi


Tue May  5 19:01:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
latent_dim = 100
img_size = 128
channels = 1

batch_size = 32
lr = 1e-4
num_epochs = 80
lambda_gp = 10
n_critic = 5


In [8]:
import os

print(os.listdir("/kaggle/working/"))


['secured_images', '.virtual_documents']


In [15]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = datasets.ImageFolder("/kaggle/input/datasets/jhanvisonii/xray-dataset/final_dataset", transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print("done")


done


In [16]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.ConvTranspose2d(100, 512, 4, 1, 0, bias=False),  # 4x4
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),  # 8x8
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),  # 16x16
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),   # 32x32
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),    # 64x64
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            nn.ConvTranspose2d(32, 1, 4, 2, 1, bias=False),     # 128x128
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)
print("done")

done


In [17]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 512, 4, 2, 1),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2),

            nn.Conv2d(512, 1, 4, 1, 0)
        )

    def forward(self, x):
        return self.net(x).view(-1)
print("done")

done


In [18]:
def gradient_penalty(critic, real, fake):
    batch_size = real.size(0)

    alpha = torch.rand(batch_size, 1, 1, 1).to(device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)

    mixed_scores = critic(interpolated)

    grad = torch.autograd.grad(
        outputs=mixed_scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True
    )[0]

    grad = grad.view(batch_size, -1)
    gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()

    return gp
    
print("done")


done


In [19]:
generator = Generator().to(device)
critic = Critic().to(device)


In [20]:
opt_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.0, 0.9))
opt_C = optim.Adam(critic.parameters(), lr=lr, betas=(0.0, 0.9))
print("ok")

ok


In [13]:
for epoch in range(num_epochs):

    for i, (real_imgs, _) in enumerate(dataloader):

        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)

        # ---------------------
        # Train Critic
        # ---------------------
        for _ in range(n_critic):

            z = torch.randn(batch_size, latent_dim, 1, 1).to(device)
            fake_imgs = generator(z)

            critic_real = critic(real_imgs)
            critic_fake = critic(fake_imgs.detach())

            gp = gradient_penalty(critic, real_imgs, fake_imgs)

            loss_C = -(torch.mean(critic_real) - torch.mean(critic_fake)) + lambda_gp * gp

            opt_C.zero_grad()
            loss_C.backward()
            opt_C.step()

        # ---------------------
        # Train Generator
        # ---------------------
        z = torch.randn(batch_size, latent_dim, 1, 1).to(device)
        fake_imgs = generator(z)

        loss_G = -torch.mean(critic(fake_imgs))

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss C: {loss_C.item():.4f}, Loss G: {loss_G.item():.4f}")


Epoch [1/80] Loss C: -0.2336, Loss G: -0.2125
Epoch [2/80] Loss C: -0.3431, Loss G: 0.2917
Epoch [3/80] Loss C: -0.2745, Loss G: -0.0018
Epoch [4/80] Loss C: -0.2467, Loss G: 0.0089
Epoch [5/80] Loss C: -0.2711, Loss G: -0.0846
Epoch [6/80] Loss C: -0.2479, Loss G: -0.2019
Epoch [7/80] Loss C: -0.2182, Loss G: -0.0785
Epoch [8/80] Loss C: -0.2364, Loss G: 0.3242
Epoch [9/80] Loss C: -0.1865, Loss G: 0.0565
Epoch [10/80] Loss C: -0.2109, Loss G: 0.2680
Epoch [11/80] Loss C: -0.1913, Loss G: 0.2313
Epoch [12/80] Loss C: -0.2163, Loss G: 0.3251
Epoch [13/80] Loss C: -0.2221, Loss G: 0.2673
Epoch [14/80] Loss C: -0.2033, Loss G: 0.5327
Epoch [15/80] Loss C: -0.1926, Loss G: 0.5626
Epoch [16/80] Loss C: -0.2142, Loss G: 0.6812
Epoch [17/80] Loss C: -0.2635, Loss G: 0.7765
Epoch [18/80] Loss C: -0.2180, Loss G: 0.8944
Epoch [19/80] Loss C: -0.2186, Loss G: 0.8062
Epoch [20/80] Loss C: -0.1971, Loss G: 0.9368
Epoch [21/80] Loss C: -0.1950, Loss G: 0.7293
Epoch [22/80] Loss C: -0.1535, Loss G:

In [27]:
torch.save(generator.state_dict(), "/kaggle/working/generator_wgan.pth")


In [21]:
generator = Generator().to(device)

generator.load_state_dict(
    torch.load("/kaggle/input/datasets/jhanvisonii/generator/generator_wgan.pth", map_location=device)
)

generator.eval()


Generator(
  (net): Sequential(
    (0): ConvTranspose2d(100, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (13): BatchNorm2d(3

In [22]:
import os
from torchvision.utils import save_image

save_dir = "/kaggle/working/generated"
os.makedirs(save_dir, exist_ok=True)

generator.eval()

num_images = 200
batch_size = 20

def enhance_xray(img_tensor):
    img = img_tensor.squeeze().cpu().numpy()

    # Normalize to 0-255
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = (img * 255).astype(np.uint8)

    # CLAHE (contrast boost)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    img = clahe.apply(img)

    # Edge detection
    edges = cv2.Canny(img, 40, 120)

    # Blend edges
    img = cv2.addWeighted(img, 0.85, edges, 0.15, 0)

    # Sharpen
    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    img = cv2.filter2D(img, -1, kernel)
    img = cv2.bilateralFilter(img, 9, 75, 75)

    return img

with torch.no_grad():
    for i in range(0, num_images, batch_size):

        z = torch.randn(batch_size, latent_dim, 1, 1).to(device) * 0.5
        fake_imgs = generator(z)
        fake_imgs = torch.nn.functional.interpolate(fake_imgs, size=(256,256), mode='bilinear')


        for j in range(fake_imgs.size(0)):

            # save raw
            save_image(
                fake_imgs[j],
                f"{save_dir}/raw_{i+j}.png",
                normalize=True
            )

            # enhance
            enhanced = enhance_xray(fake_imgs[j])

            cv2.imwrite(
                f"{save_dir}/enhanced_{i+j}.png",
                enhanced
            )

print("Images Generated + Enhanced Successfully")


Images Generated + Enhanced Successfully


In [23]:
import shutil

shutil.make_archive(
    '/kaggle/working/generated_imges_final',  # zip file name
    'zip',
    '/kaggle/working/generated'      # folder to zip
)

print("ZIP file created!")


ZIP file created!


In [24]:
import cv2

def add_watermark(image_path, output_path, text="GEN_AI_XRAY"):
    img = cv2.imread(image_path)

    h, w = img.shape[:2]

    # position
    position = (w - 250, h - 20)

    # add watermark
    cv2.putText(
        img,
        text,
        position,
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (200, 200, 200),
        1,
        cv2.LINE_AA
    )

    cv2.imwrite(output_path, img)

print("task complete")


task complete


In [25]:
import hashlib

def generate_hash(image_path):
    with open(image_path, "rb") as f:
        bytes = f.read()
        hash_val = hashlib.sha256(bytes).hexdigest()
    return hash_val

print("task complete")

task complete


In [26]:
import os

input_folder = "/kaggle/working/generated"
secure_folder = "/kaggle/working/secured_images"

os.makedirs(secure_folder, exist_ok=True)

hash_records = {}

for file in os.listdir(input_folder):
    if file.endswith(".png"):

        input_path = os.path.join(input_folder, file)
        output_path = os.path.join(secure_folder, file)

        # watermark
        add_watermark(input_path, output_path)

        # hash
        hash_val = generate_hash(output_path)

        hash_records[file] = hash_val

print("Security Applied Successfully")


Security Applied Successfully


In [27]:
import json

with open("/kaggle/working/hash_records.json", "w") as f:
    json.dump(hash_records, f, indent=4)

print("task done")


task done


In [28]:
import shutil

shutil.make_archive(
    '/kaggle/working/secured_images',
    'zip',
    '/kaggle/working/secured_images'
)
print("zip file created")

zip file created
